In [1]:
from ollama import chat
from ollama import ChatResponse

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_ollama.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [2]:
# Checking access to local model
response: ChatResponse = chat(
  model='phi4-mini',
  messages=[
    {
      'role': 'user',
      'content': 'In 50 words, what is llm?',
    },
  ],
)

print(response.message.content)

LLM stands for Large Language Model, a sophisticated AI designed to understand, generate, and interpret human language, capable of performing tasks such as translation, summarization, and conversation.


# To load pdf

In [3]:
# Load PDF
pdf_path = "docs/th_intro.pdf"

reader = PdfReader(pdf_path)
num_p = len(reader.pages)
page = reader.pages[0]
text = page.extract_text()

In [4]:
text

'Chapter 1\nIntroduction\nThe evolution of wireless communication technology has progressed through multiple gen-\nerations, each introducing significant advancements in data rates, coverage, and func-\ntionality. Each generation has had significant impact on the society by enabling global\nconnectivity, fostering new business models, and driving economic growth. Emerging ap-\nplication use cases impose increasingly demanding requirements, which must be met by\nthe enabling technologies of future wireless communication systems [1] [2].\nThe requirements for Beyond 5G (B5G) are well-known, including peak data rates in\nthe terabit-per-second (Tb/s) range, sub-millisecond latencies, centimeter-level location\naccuracy, and improved spectrum efficiency, among others [3]. The capabilities defined by\nInternational Mobile Telecommunications for 2030 and beyond (IMT-2030) set ambitious\ntargets [1] to address the evolving needs of society, as illustrated in Figure 1.1. The\nupcoming 6G era i

# To create Chunks -> Vector DB

In [5]:
# Split text into chunks
page_documents = []
for page_num, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    if text.strip():
        page_documents.append(
            Document(
                page_content=text,
                metadata={"page": page_num},
            )
        )

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)
chunk_documents = text_splitter.split_documents(page_documents)

print(f"Loaded {len(reader.pages)} pages")
print(f"Created {len(chunk_documents)} chunks")

Loaded 13 pages
Created 37 chunks


In [6]:
# Create vector database

embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma.from_documents(
    documents=chunk_documents,
    embedding=embeddings,
    persist_directory="./chroma_db",
    collection_name="pdf_coll",
)

print("Vector database created")

Vector database created


# RAG Chain

In [7]:
# Build the RAG chain
llm = ChatOllama(model="phi4-mini", temperature=0)

In [8]:
# Query prompt template
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Answer the question using ONLY the supplied context. "
        "If the context doesn't contain the answer, say you don't know.\n\n"
        "Context:\n{context}",
    ),
    ("human", "{input}"),
])

# Set up retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)

# Questions regarding PDF

In [9]:
result = rag_chain.invoke({"input": "In 80 words, what is this PDF about?"})
print(result["answer"])

The PDF discusses the evolution of wireless communication technology through various generations, highlighting advancements in data rates, coverage, and functionality. It mentions the impact on society, including global connectivity and economic growth. The document also outlines the requirements for Beyond 5G (B5G), such as terabit-per-second data rates, sub-millisecond latencies, centimeter-level location accuracy, and improved spectrum efficiency. These requirements are part of the International Mobile Telecommunications for 2030 and beyond (IMT-2030) standards.


In [10]:
# inspect the retrieved chunks
for doc in result["context"]:
    print(doc.page_content)

space, the action space, and the overall goal [15]. The schematics of the framework
is visualized in Figure 1.4. A learning agent observes the state of the environment
and takes an action which is capable of changing the next state. For every ac-
tion the agent performs, it receives a corresponding reward. In RL, the goal of
the agent is to increase the cumulative reward with every action it takes, building
knowledge/experience which is exploited at later time steps. The advancement of
RL has resulted in many algorithm variants of notable importance including Deep
RL (DRL), Deep Q-Network (DQN), Double Deep Q-Learning (DDQN), Advantage
Actor-Critic (A2C) [16].
Deep Learning (DL) is a subset of ML which uses multi-layer neural networks to solve
classification or regression tasks [17]. The Multi-Layer Perceptron (MLP) can be regarded
as universal function approximators that define a mapping from an input vector/features
4 1.1. AI Taxonomy4 1.1. AI Taxonomy4 1.1. AI Taxonomy
chical Cluste

In [11]:
result = rag_chain.invoke({"input": "How many journals are listed under publications?"})
print(result["answer"])

Two journals are listed under publications.


In [12]:
result = rag_chain.invoke({"input": "Is there a mention of patent?"})
print(result["answer"])

The context provided does not mention any patents.
